# haRdQualizer — Goth character generator (Animagine XL, Colab T4)

Generates **original** anime/illustration goth-girl characters for the
Alt/Gothic visualizer, removes the background, and packages transparent PNGs
that drop straight into `assets/characters/`.

Uses **Animagine XL 3.1** (a strong anime SDXL fine-tune) instead of raw SDXL
base — far better stylized characters, framing and hands.

### Scope / ethics
Makes **original, fictional characters** in a goth *aesthetic* (makeup, horns,
spiked chokers, fishnet, platform boots, gold jewelry). It does **not** copy the
likeness of any real person — use reference photos only as a style moodboard.

### How to run
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Run the cells top to bottom.
3. **If cell 3 asserts Pillow >= 12:** Runtime -> Restart session, then run from
   cell 3 (skip cell 2). `rembg` upgrades Pillow and breaks `diffusers`; the pin
   + restart fixes it.
4. The last cell downloads `characters.zip`. Extract it into
   `haRdQualizer/assets/characters/` so you get `goth_a/body.png`, etc.
5. Don't like a character? Change its `seed` (or `desc`) in cell 5 and re-run
   cells 5-6.

In [ ]:
# 1. Check the GPU (expect a Tesla T4, ~15GB).
!nvidia-smi

In [ ]:
# 2. Install dependencies (~2-3 min).
#
# IMPORTANT: rembg pulls in Pillow 12, which breaks Colab's diffusers
# ("ImportError: cannot import name '_Ink' from 'PIL._typing'"). We pin
# "pillow<12" in the SAME command. The cudf / cuml / numba / scikit-image
# warnings printed by pip are harmless (unused RAPIDS libs).
!pip install -q diffusers transformers accelerate safetensors rembg onnxruntime "pillow<12"

print("\n*** If pip changed Pillow: Runtime > Restart session, then run from cell 3.")

In [ ]:
# 3. Guard: confirm a diffusers-compatible Pillow before importing anything.
import PIL
major = int(PIL.__version__.split(".")[0])
print("Pillow", PIL.__version__)
assert major < 12, (
    "Pillow >= 12 is loaded and will break diffusers. "
    "Do Runtime > Restart session, then re-run from this cell (skip cell 2)."
)
print("Pillow OK")

In [ ]:
# 4. Load an ANIME SDXL checkpoint ON THE GPU (not system RAM).
#
# We use Animagine XL 3.1 — a strong, diffusers-native anime fine-tune. It is
# dramatically better than raw SDXL base for stylized characters, respects the
# "full body" framing tag, and handles hands far better. (To try another style
# later, swap MODEL_ID, e.g. "cagliostrolab/animagine-xl-4.0".)
#
# Loaded straight onto the T4's 15GB VRAM (.to cuda) so the ~12.7GB system RAM
# never overflows — that was the earlier "session crashed, used all RAM" cause.
import gc
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

MODEL_ID = "cagliostrolab/animagine-xl-3.1"

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    low_cpu_mem_usage=True,
)
# Euler a is the recommended sampler for Animagine.
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

gc.collect(); torch.cuda.empty_cache()
print("pipeline ready on", pipe.device)
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

**If memory is still tight**
- *"Session crashed, used all RAM"* again -> Runtime -> **Restart session** to
  clear the previous crash's leftovers, then run cell 4 **once** (running it
  twice loads the model twice).
- *CUDA out of memory* -> lower `WIDTH, HEIGHT` to `768, 1024` in cell 5, or set
  `STEPS = 24`.

**Want different art?** Change `MODEL_ID` in cell 4:
- `cagliostrolab/animagine-xl-4.0` — newer Animagine.
- `stabilityai/stable-diffusion-xl-base-1.0` — plain SDXL (also add
  `variant="fp16"`).
For Pony / Illustrious / Civitai checkpoints, download the `.safetensors` and
load with `StableDiffusionXLPipeline.from_single_file(path, ...)`.

In [ ]:
# 5. Configuration — anime prompts tuned for Animagine XL 3.1.
#
# Animagine responds to Danbooru-style tags. The quality suffix + the "full
# body, white background" framing fix the earlier problems (cropped portraits,
# busy background). Keep a fixed `seed` per character for repeatable results.

# Animagine's recommended quality suffix.
QUALITY = "masterpiece, best quality, very aesthetic, absurdres"

# Framing that forces a clean full-length standee on a white background.
FRAMING = (
    "1girl, solo, standing, full body, looking at viewer, arms at sides, "
    "feet visible, simple background, white background"
)

STYLE = "{desc}, " + FRAMING + ", " + QUALITY

# Animagine's recommended negative + framing/anatomy guards.
NEG = (
    "nsfw, nude, lowres, (bad), text, error, fewer, extra, missing, "
    "worst quality, jpeg artifacts, low quality, watermark, unfinished, "
    "displeasing, chromatic aberration, signature, extra digits, "
    "artistic error, username, scan, bad hands, bad anatomy, extra limbs, "
    "cropped, head out of frame, multiple girls, 2girls, portrait, close-up"
)

CHARACTERS = [
    {
        "name": "goth_a",
        "seed": 12345,
        "desc": "goth girl, long straight black hair, blunt bangs, small red "
                "demon horns, spiked black choker, black gothic dress, detached "
                "sleeves, fishnet legwear, black platform boots, pale skin, "
                "black lipstick, gothic eye makeup",
    },
    {
        "name": "goth_b",
        "seed": 5544,
        "desc": "goth girl, very long wavy black hair, gold head chain, gold "
                "jewelry, black corset dress with gold accents, lace gloves, "
                "fishnet legwear, black platform boots, pale skin, dark lips",
    },
    {
        "name": "goth_c",
        "seed": 909090,
        "desc": "goth girl, black hair with dark green streaks, studded choker, "
                "black band t-shirt, black tulle skirt, torn fishnet legwear, "
                "black platform boots, green eyes, dark gothic makeup",
    },
]

# Animagine-friendly settings (Euler a, CFG ~6).
STEPS = 28
GUIDANCE = 6.0
WIDTH, HEIGHT = 832, 1216   # tall portrait -> full standing figure

In [ ]:
# 6. Generate one clean full-body image per character.
from pathlib import Path

RAW_DIR = Path("/content/raw")
RAW_DIR.mkdir(exist_ok=True)
raw_images = {}

def generate(prompt, seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    img = pipe(
        prompt=prompt, negative_prompt=NEG,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE,
        width=WIDTH, height=HEIGHT, generator=g,
    ).images[0]
    torch.cuda.empty_cache()   # release transient VRAM between characters
    return img

for c in CHARACTERS:
    prompt = STYLE.format(desc=c["desc"])
    print(f"generating {c['name']} (seed {c['seed']}) ...")
    img = generate(prompt, c["seed"])
    img.save(RAW_DIR / f"{c['name']}.png")
    raw_images[c["name"]] = img
print("done")

In [ ]:
# 7. Preview the raw renders. Re-run cells 5-6 with new seeds for any you dislike.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(raw_images), figsize=(4 * len(raw_images), 7))
if len(raw_images) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, raw_images.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.show()

In [ ]:
# 8. Remove the background with the ANIME matting model (isnet-anime).
#
# The default rembg model (u2net) eats pale skin / light hair on a WHITE
# background — that is why a pale character can come out headless. isnet-anime
# is trained on anime art and segments the whole character cleanly regardless
# of background colour. (First call downloads the ~170 MB model once.)
from rembg import remove, new_session

anime_session = new_session("isnet-anime")

OUT_DIR = Path("/content/characters")
OUT_DIR.mkdir(exist_ok=True)

for c in CHARACTERS:
    cut = remove(raw_images[c["name"]], session=anime_session)
    d = OUT_DIR / c["name"]
    d.mkdir(parents=True, exist_ok=True)
    cut.save(d / "body.png")
    print("saved", d / "body.png")

## Optional: pose variants (hands_up / dance)

The rig already animates the single `body.png` with sway / breathe / hop. For
distinct pose art, generate variants below with img2img. They save as
`hands_up.png` / `dance.png`; the rig swaps to them when that pose is active.

In [ ]:
# 9. (Optional) Pose variants via img2img. Skip if you only want body.png.
#
# Reuses the base pipeline's components (weights already on the GPU), so it adds
# no extra VRAM/RAM. Uses the same isnet-anime session from cell 8.
#
# NOTE: STRENGTH 0.5 barely changes the pose. Raise it to ~0.65-0.72 if you want
# arms actually raised for "hands_up" — at the cost of the character drifting a
# bit from body.png. If the variants look the same as body.png, just skip this
# cell; the rig already animates body.png with sway / breathe / hop.
from diffusers import StableDiffusionXLImg2ImgPipeline

img2img = StableDiffusionXLImg2ImgPipeline(**pipe.components)

POSE_PROMPTS = {
    "hands_up": "arms raised overhead, both hands up high, \\(reaching up\\)",
    "dance": "dancing, arms out to the sides, dynamic pose",
}
STRENGTH = 0.68  # higher than before so the pose actually changes

for c in CHARACTERS:
    base = raw_images[c["name"]]
    for pose, extra in POSE_PROMPTS.items():
        prompt = c["desc"] + ", " + extra + ", " + FRAMING + ", " + QUALITY
        g = torch.Generator(device="cuda").manual_seed(c["seed"])
        out = img2img(
            prompt=prompt, negative_prompt=NEG, image=base,
            strength=STRENGTH, num_inference_steps=STEPS,
            guidance_scale=GUIDANCE, generator=g,
        ).images[0]
        torch.cuda.empty_cache()
        cut = remove(out, session=anime_session)
        (OUT_DIR / c["name"]).mkdir(parents=True, exist_ok=True)
        cut.save(OUT_DIR / c["name"] / f"{pose}.png")
        print("saved", c["name"], pose)

In [ ]:
# 10. Package everything and download.
import shutil
from google.colab import files

shutil.make_archive("/content/characters", "zip", "/content/characters")
print("Extract characters.zip into haRdQualizer/assets/characters/")
files.download("/content/characters.zip")